# Quickstart: Querying PDF With Astra and LangChain

### A question-answering demo using Astra DB and LangChain, powered by Vector Search

#### Pre-requisites:

You need a **_Serverless Cassandra with Vector Search_** database on [Astra DB](https://astra.datastax.com) to run this demo. As outlined in more detail [here](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html#_prepare_for_using_your_vector_database), you should get a DB Token with role _Database Administrator_ and copy your Database ID: these connection parameters are needed momentarily.

You also need an [OpenAI API Key](https://cassio.org/start_here/#llm-access) for this demo to work.

#### What you will do:

- Setup: import dependencies, provide secrets, create the LangChain vector store;
- Run a Question-Answering loop retrieving the relevant headlines and having an LLM construct the answer.

Install the required dependencies:

In [1]:
!pip install -qU \
    langchain \
    langchain-astradb \
    langchain-groq \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.9/360.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.0/639.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Import the packages you'll need:

In [2]:
from pypdf import PdfReader

from langchain_astradb import AstraDBVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
from google.colab import userdata

ASTRA_DB_APPLICATION_TOKEN = userdata.get("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_API_ENDPOINT = userdata.get("ASTRA_DB_API_ENDPOINT")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not ASTRA_DB_APPLICATION_TOKEN:
    raise ValueError("ASTRA_DB_APPLICATION_TOKEN not found in Colab Secrets.")

if not ASTRA_DB_API_ENDPOINT:
    raise ValueError("ASTRA_DB_API_ENDPOINT not found in Colab Secrets.")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

print("All credentials loaded successfully.")

All credentials loaded successfully.


### Setup

In [4]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()

pdf_name = next(iter(uploaded))

pdfreader = PdfReader(pdf_name)

print(f"Uploaded PDF: {pdf_name}")
print(f"Number of pages: {len(pdfreader.pages)}")

Saving AI_Healthcare_Long_Test_Document.pdf to AI_Healthcare_Long_Test_Document.pdf
Uploaded PDF: AI_Healthcare_Long_Test_Document.pdf
Number of pages: 7


In [5]:
documents = []

for page_number, page in enumerate(pdfreader.pages, start=1):
    content = page.extract_text()

    if content and content.strip():
        documents.append(
            Document(
                page_content=content,
                metadata={
                    "source": pdf_name,
                    "document_id": pdf_name,
                    "page": page_number
                }
            )
        )

print(f"Extracted {len(documents)} pages from {pdf_name}.")

Extracted 7 pages from AI_Healthcare_Long_Test_Document.pdf.


In [6]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


Create your LangChain vector store ... backed by Astra DB!

In [7]:
astra_vector_store = AstraDBVectorStore(
    collection_name="pdf_qa_demo",
    embedding=embedding,
    token=ASTRA_DB_APPLICATION_TOKEN,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
)

print("Astra DB vector store initialized successfully.")

Astra DB vector store initialized successfully.


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

split_documents = text_splitter.split_documents(documents)

print(f"Created {len(split_documents)} chunks.")

Created 21 chunks.


In [9]:
print("Removing previous document data from Astra DB...")

astra_vector_store.clear()

print("Previous document data removed successfully.")

Removing previous document data from Astra DB...
Previous document data removed successfully.


In [10]:
inserted_ids = astra_vector_store.add_documents(
    split_documents
)

print(f"Inserted {len(inserted_ids)} chunks into Astra DB.")
print(f"Current document: {pdf_name}")

Inserted 21 chunks into Astra DB.
Current document: AI_Healthcare_Long_Test_Document.pdf


### Load the dataset into the vector store



In [11]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

print("Groq LLM initialized successfully.")

Groq LLM initialized successfully.


In [12]:
prompt = ChatPromptTemplate.from_template("""
You are a document question-answering assistant.

Your task is to answer the user's question based strictly on the
provided document context.

IMPORTANT RULES:

1. Carefully read ALL the provided context before answering.
2. The answer may be stated directly or indirectly in the context.
3. Use information from multiple context sections if necessary.
4. Do not use outside knowledge.
5. Do not invent or assume facts that are not supported by the context.
6. If the context contains enough information to answer the question,
   provide the answer clearly and directly.
7. Only say that the answer cannot be found if the provided context
   genuinely contains no relevant information.

Context:
{context}

Question:
{question}

Answer:
""")

In [13]:
rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully.")

RAG chain created successfully.


In [14]:
def retrieve_context(query, k=4):
    return astra_vector_store.similarity_search_with_score(
        query,
        k=k
    )


def generate_answer(query):
    results = retrieve_context(query, k=4)

    context = "\n\n".join(
        [
            f"[Page {doc.metadata.get('page')}]\n{doc.page_content}"
            for doc, score in results
        ]
    )

    answer = rag_chain.invoke({
        "context": context,
        "question": query
    })

    return answer, results

In [15]:
query_text = "What is the fiscal deficit in BE 2026-27?"

results = astra_vector_store.similarity_search_with_score(
    query_text,
    k=4
)

print("RETRIEVED DOCUMENTS")
print("=" * 70)

for i, (doc, score) in enumerate(results, start=1):
    print(f"\nDocument {i}")
    print(f"Score: {score:.4f}")
    print(f"Page: {doc.metadata.get('page')}")
    print(doc.page_content[:1000])

RETRIEVED DOCUMENTS

Document 1
Score: 0.5461
Page: 4
follow-up action was recommended?'
16. Proposed System Architecture
The proposed system has five major stages. First, documents are ingested and converted into
machine-readable text. Second, the text is " cleaned and divided into chunks while preserving section
information. Third, embeddings are generated for the chunks and stored in a vector " index. Fourth, a
retrieval component selects relevant chunks for a user question. Fifth, a language model generates an
answer using the " retrieved context. The interface can display the answer together with the relevant
source passages so the user can verify the result.
17. Example Workflow
Suppose a discharge document contains a 2,500-word description of a patient's hospital stay. The
summarization module may produce a " summary organized into admission reason, major findings,
treatment, medications, progress, and follow-up. Later, a user asks, 'Which " follow-up appointment was

Document 2

In [16]:
test_questions = [
    "What is the fiscal deficit in BE 2026-27?",
    "What are the estimated non-debt receipts for 2026-27?",
    "What is the estimated total expenditure for 2026-27?",
    "What are the Centre's net tax receipts for 2026-27?",
    "What is the debt-to-GDP ratio target for 2030-31?",
    "Who is the Prime Minister of Japan?"
]

for question in test_questions:

    results = astra_vector_store.similarity_search_with_score(
        question,
        k=4
    )

    context = "\n\n".join(
        [
            f"[Page {doc.metadata.get('page')}]\n{doc.page_content}"
            for doc, score in results
        ]
    )

    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("ANSWER:", answer)


QUESTION: What is the fiscal deficit in BE 2026-27?
ANSWER: The provided document does not contain any information about the fiscal deficit for BE 2026‑27.

QUESTION: What are the estimated non-debt receipts for 2026-27?
ANSWER: The provided document does not contain any information about estimated non‑debt receipts for 2026‑27.

QUESTION: What is the estimated total expenditure for 2026-27?
ANSWER: The provided context does not contain any information about an estimated total expenditure for 2026‑27.

QUESTION: What are the Centre's net tax receipts for 2026-27?
ANSWER: The provided document does not contain information about the Centre's net tax receipts for 2026‑27.

QUESTION: What is the debt-to-GDP ratio target for 2030-31?
ANSWER: The provided document does not contain any information about a debt‑to‑GDP ratio target for 2030‑31.

QUESTION: Who is the Prime Minister of Japan?
ANSWER: The provided document does not contain any information about the Prime Minister of Japan.


### Run the QA cycle

Simply run the cells and ask a question -- or `quit` to stop. (you can also stop execution with the "▪" button on the top toolbar)

Here are some suggested questions:
- _What is the current GDP?_
- _How much the agriculture target will be increased to and what the focus will be_


In [17]:
first_question = True

while True:

    if first_question:
        query_text = input(
            "\nEnter your question (or type 'quit' to exit): "
        ).strip()
    else:
        query_text = input(
            "\nWhat's your next question (or type 'quit' to exit): "
        ).strip()

    if query_text.lower() == "quit":
        print("\nExiting...")
        break

    if not query_text:
        continue

    first_question = False

    print("\n" + "=" * 70)
    print(f"QUESTION: {query_text}")
    print("=" * 70)

    # Retrieve relevant documents
    results = astra_vector_store.similarity_search_with_score(
        query_text,
        k=4
    )

    # Build context
    context = "\n\n".join(
        [
            f"[Page {doc.metadata.get('page')}]\n{doc.page_content}"
            for doc, score in results
        ]
    )

    # Generate answer
    answer = rag_chain.invoke({
        "context": context,
        "question": query_text
    })

    print("\nANSWER:")
    print(answer)

    # Show retrieved sources
    print("\n" + "-" * 70)
    print("RELEVANT DOCUMENTS")
    print("-" * 70)

    for i, (doc, score) in enumerate(results, start=1):

        print(f"\nDocument {i}")
        print(f"Similarity score: {score:.4f}")
        print(f"Page: {doc.metadata.get('page')}")
        print(f"Content: {doc.page_content[:300]}...")


Enter your question (or type 'quit' to exit): What are the major categories of AI applications in healthcare?

QUESTION: What are the major categories of AI applications in healthcare?

ANSWER:
The document lists four major categories of AI applications in healthcare:

1. **Medical imaging systems** – AI that analyzes X‑rays, CT scans, MRIs, ultrasounds, pathology slides, etc.  
2. **Predictive models** – AI that estimates the probability of events such as hospital readmission or patient deterioration.  
3. **Natural‑language‑processing (NLP) systems** – AI that summarizes clinical notes, extracts key entities, organizes information, and answers questions about documents.  
4. **Administrative systems** – AI that assists with scheduling, coding, billing workflows, and resource planning.

----------------------------------------------------------------------
RELEVANT DOCUMENTS
----------------------------------------------------------------------

Document 1
Similarity score: 0.8358
Pa